# Multi-region model: distributed predictions (Sec. 3.4; Fig. 7, A11–A16)

Applies the multi-region model (zero-shot) and its fine-tuned versions (from `3.2 Pooled_model`) to the glacier grids of the stake glaciers, and compares the predicted maps and elevation gradients with the stakes.

| Paper figure | Content | Saved as (in `figures/paperTF/`) |
|---|---|---|
| Fig. 7 | Zero-shot vs fine-tuned MB maps of eight glaciers, pixel scatter, and elevation gradients | `fig7_application.png` |
| Fig. A11 | Elevation gradients, Switzerland | `appendix/A11_gradients_CH.png` |
| Fig. A12 | Elevation gradients, France, Italy and Austria | `appendix/A12_gradients_CEU.png` |
| Fig. A13 | Elevation gradients, Norway and Svalbard | `appendix/A13_gradients_NOR_SJM.png` |
| Fig. A14 | Elevation gradients, Iceland | `appendix/A14_gradients_ISL.png` |
| Fig. A15 | Elevation gradients, Central Asia | `appendix/A15_gradients_CA.png` |
| Fig. A16 | Elevation gradients, Alaska | `appendix/A16_gradients_ALA.png` |

**Steps**
1. Load the stake datasets and monthly inputs, and split Central Asia, Alaska and Italy/Austria into sub-regions.
2. Load the multi-region model and the fine-tuned models (`full` strategy, 10% local data) of each region.
3. Predict annual MB on the grids of all stake glaciers of the regions in `REGIONS_TO_PREDICT` (empty by default: the predictions already on disk are used).
4. Make the figures.

**Requires** the multi-region and fine-tuned models of `3.2 Pooled_model` (`cfg.dataPath/cache/TF_Europe/FD_MODEL/models/`) and, for new predictions, the cached grid datasets in `cfg.dataPath/RGI_v6/<region>/grid_ds_cache/` (`1.4 Create_grids`). Predictions are saved in `cfg.dataPath/cache/pooled_pred/predictions_transformer/`. The functions used here live in `scripts/plotting/pooled_pred.py` and `scripts/plotting/pooled_pred_figures.py`.

## Setting up:

In [ ]:
import os

# Silence KeOps (used by geomloss for Sinkhorn distances) before it is imported
os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

import torch

# Initialise CUDA before the other imports: changing this order can change the
# GPU results slightly (~1e-7)
if torch.cuda.is_available():
    torch.cuda.init()

import sys
import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import massbalancemachine as mbm

sys.path.append(os.path.join(os.getcwd(), '../../'))  # Add root of repo to import MBM
# Shared config (paths, MONTHLY_COLS, STATIC_COLS, ...), model settings
# (PARAMS_TRANSFORMER, N_EPOCHS, MODEL_DATE) and helpers
from papers.Transformer_TF.scripts.config_TF_Europe import *
from papers.Transformer_TF.scripts.config_models import *
from papers.Transformer_TF.scripts.dataset import *
from papers.Transformer_TF.scripts.plotting import *
from papers.Transformer_TF.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

# Call config file (data paths, seed, etc.)
cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)  # reproducibility
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

# Use the GPU if one is visible
device = torch.device("cuda" if torch.cuda.device_count() > 0 else "cpu")
print(f"Device: {device}")

# Cache folder of the multi-region experiments (datasets, models)
BASE_DIR = Path(cfg.dataPath) / path_cache / "FD_MODEL"
CACHE_DIR = BASE_DIR / "datasets"
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Base directory for data: {BASE_DIR}")

# Folder of the predicted grids
PRED_BASE = os.path.join(cfg.dataPath, "cache/pooled_pred/predictions_transformer")

os.makedirs("figures/paperTF/appendix", exist_ok=True)

## Data:
With `run_flag_by_code = False` (default), the monthly data are loaded from `WGMS/CrossRegional/monthly_cache/` instead of being recomputed.

In [ ]:
SOURCE_REGIONS = ['CH', 'NOR', 'ISL', "FR"]  # training regions of the multi-region model
TARGET_REGIONS = ['IT_AT', 'SJM', 'CENTRALASIA', 'ALA']
TARGET_REGIONS_SUB = ['IT', 'AT', 'SJM', 'ALA', 'CA_3', 'CA_4']

data = load_pooled_experiment_data(cfg, SOURCE_REGIONS, TARGET_REGIONS,
                                   TARGET_REGIONS_SUB)
dfs = data["dfs"]
monthly_cache = data["monthly_cache"]
res_xreg_by_source = data["res_xreg_by_source"]
months_head_pad = data["months_head_pad"]
months_tail_pad = data["months_tail_pad"]

## Models:
### Multi-region model:

In [ ]:
assets_full = build_pooled_assets(
    res_xreg_by_source,
    SOURCE_REGIONS,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    cache_path=CACHE_DIR / "assets_foundation_100pct.joblib",
)

model_pooled, ds_pooled_scaler = load_pooled_model(
    cfg,
    device,
    assets_full,
    params=PARAMS_TRANSFORMER,
    models_dir=BASE_DIR / "models" / "foundation",
    epochs=N_EPOCHS,
    date=MODEL_DATE,
)

### Fine-tuned models:
Full fine-tuning on 10% of the measurements of each region, `N_REPEATS` repeats (only repeat 0 is used for the maps). Swiss, French, Italian and Austrian glaciers all use the model fine-tuned on Switzerland. Models are loaded from `FD_MODEL/models/finetuned/`, or trained if missing.

In [ ]:
# FT_FRAC and the number of repeats are set in scripts/config_models.py
N_REPEATS = FT_N_REPEATS

# RGI region (key) -> region whose data is used for fine-tuning
RGI_TO_FT_REGION = {
    "06": "ISL",
    "07": "SJM",
    "08": "NOR",
    "01": "ALA",
    "11": "CH",
    "13_CA_3": "CA_3",
    "13_CA_4": "CA_4"
}

ft_models_by_rgi = {}
for rgi_id, ft_code in RGI_TO_FT_REGION.items():
    print(f"\n{'='*60}\nRGI {rgi_id} ({ft_code})")
    ft_models_by_rgi[rgi_id] = train_or_load_ft_models_for_region(
        region_code=ft_code,
        monthly_cache=monthly_cache,
        model_pooled=model_pooled,
        models_dir_ft=BASE_DIR / "models" / "finetuned",
        strategy="full",
        n_repeats=N_REPEATS,
        model_date=MODEL_DATE,
        cfg=cfg,
        device=device,
        best_params=PARAMS_TRANSFORMER,
        ds_pooled_scaler=ds_pooled_scaler,
        ft_frac=FT_FRAC,
        force_retrain=False,
        CACHE_DIR=CACHE_DIR)
    print(f"  -> {len(ft_models_by_rgi[rgi_id])} models ready")

# Fine-tuned models per prediction key "<RGI region>_<region code>"
ft_models_for_pred = build_ft_models_for_pred(ft_models_by_rgi, RGI_TO_FT_REGION)

## Predictions:
### Stake glaciers:
One config per stake glacier: RGI IDs, RGI folder, key of its fine-tuned model and stake data.

In [ ]:
WANTED_REGION_CODES = ["CH", "FR", "ISL", "ALA", "NOR", "CA_3", "CA_4", "SJM", "IT", "AT"]

ALL_REGION_CONFIGS, ALL_GLACIER_CONFIG = build_all_region_configs(
    WANTED_REGION_CODES, monthly_cache, dfs, ft_models_for_pred)

### Predict:
Annual zero-shot and fine-tuned (repeat 0) prediction grids of all stake glaciers of each region in `REGIONS_TO_PREDICT`. This is slow: run one region at a time and restart the kernel in between for a clean GPU state. Leave the list empty to only make the figures from the predictions on disk.

In [ ]:
REGIONS_TO_PREDICT = []  # e.g. ["CH"], ["IT", "AT"]; any of WANTED_REGION_CODES

for region_code in REGIONS_TO_PREDICT:
    run_region(region_code,
               ALL_REGION_CONFIGS,
               cfg,
               device,
               model_pooled,
               ds_pooled_scaler,
               ft_models_for_pred,
               PRED_BASE,
               months_head_pad,
               months_tail_pad)

## Figures:
### Fig. 7: zero-shot vs fine-tuned maps
Each row holds up to two (glacier, year) cells; `text_pos` sets on which side the RMSE / mean MB annotation goes. The bottom row shows the elevation gradients of all glaciers.

In [ ]:
YEAR = 2021
MAP_ROWS = [
    [
        {"glacier_name": "Argentiere", "year": YEAR, "text_pos": "right"},
        {"glacier_name": "aletsch", "year": YEAR},
    ],
    [
        {"glacier_name": "Breidamerkurjoekull", "year": YEAR},
        {"glacier_name": "Nigardsbreen", "year": 2020},
    ],
    [
        {"glacier_name": "NORDENSKIOELDBREEN", "year": YEAR, "text_pos": "right"},
        {"glacier_name": "LEMON CREEK", "year": YEAR, "text_pos": "right"},
    ],
    [
        {"glacier_name": 'TS.TUYUKSUYSKIY', "year": YEAR},
        {"glacier_name": 'Golubin', "year": YEAR},
    ],
]

fig = plot_application_figure(MAP_ROWS, ALL_GLACIER_CONFIG, PRED_BASE)
plt.show()
fig.savefig('figures/paperTF/fig7_application.png')

### Fig. A11–A16: elevation gradients of all stake glaciers

In [ ]:
# (glacier regions, number of columns, file name) per figure
GRADIENT_FIGURES = [
    (["CH"], 6, "A11_gradients_CH.png"),
    (["FR", "IT", "AT"], 6, "A12_gradients_CEU.png"),
    (["NOR", "SJM"], 6, "A13_gradients_NOR_SJM.png"),
    (["ISL"], 8, "A14_gradients_ISL.png"),
    (["CA_3", "CA_4"], 5, "A15_gradients_CA.png"),
    (["ALA"], 5, "A16_gradients_ALA.png"),
]

for region_codes, ncols, fname in GRADIENT_FIGURES:
    glaciers = [g for code in region_codes for g in ALL_REGION_CONFIGS[code]]
    fig = plot_gradients_grid(glaciers, ALL_GLACIER_CONFIG, PRED_BASE, ncols=ncols)
    plt.show()
    fig.savefig(f'figures/paperTF/appendix/{fname}')